# Genetic Algorithm Optimization with topologic_fast

This notebook demonstrates genetic algorithm (GA) optimization applied to topology and geometric problems using `topologic_fast`.

## Overview

Genetic algorithms are metaheuristic optimization algorithms inspired by natural selection. They are useful for:
- Optimizing building layouts
- Finding optimal geometric configurations
- Multi-objective design optimization

**Note:** topologicpy provides a `GA` class that wraps PyGAD. In this notebook, we use PyGAD directly with topologic_fast for geometry generation and evaluation. This demonstrates how to integrate topologic_fast with optimization libraries.

In this notebook, we'll:
1. Set up a basic GA optimization problem
2. Optimize geometric parameters using topology
3. Multi-objective optimization with Pareto fronts
4. Visualize optimization results

In [ ]:
# Import required libraries
import topologic_fast as tf
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Try to import PyGAD, provide fallback if not available
try:
    import pygad
    PYGAD_AVAILABLE = True
    print("PyGAD is available")
except ImportError:
    PYGAD_AVAILABLE = False
    print("PyGAD not installed. Install with: pip install pygad")
    print("This notebook will demonstrate the concepts without running the GA.")

## 1. Simple Optimization Example

Let's start with a simple example: optimizing the dimensions of a box (Cell) to maximize volume while keeping surface area below a threshold.

In [ ]:
def create_box_and_evaluate(width, length, height):
    """Create a box cell and return its volume and surface area."""
    # Ensure positive dimensions
    width = max(0.1, width)
    length = max(0.1, length)
    height = max(0.1, height)
    
    # Create a box cell
    box = tf.Cell.Box(0, 0, 0, width, length, height)
    
    volume = box.Volume()
    area = box.Area()
    
    return volume, area

# Test the function
vol, area = create_box_and_evaluate(2, 3, 4)
print(f"Box (2x3x4):")
print(f"  Volume: {vol}")
print(f"  Surface Area: {area}")

In [ ]:
# Define the fitness function for maximizing volume with area constraint
MAX_AREA = 100  # Maximum allowed surface area

def fitness_volume_constrained(ga_instance, solution, solution_idx):
    """Fitness function: maximize volume while keeping area <= MAX_AREA."""
    width, length, height = solution
    
    volume, area = create_box_and_evaluate(width, length, height)
    
    # Penalty for exceeding area constraint
    if area > MAX_AREA:
        penalty = (area - MAX_AREA) * 10
        fitness = volume - penalty
    else:
        fitness = volume
    
    return fitness

# Test fitness function
test_solution = [2, 3, 4]
fitness = fitness_volume_constrained(None, test_solution, 0)
print(f"Fitness for box (2x3x4): {fitness}")

In [ ]:
if PYGAD_AVAILABLE:
    # Gene space: dimensions between 0.5 and 10
    gene_space = [{"low": 0.5, "high": 10.0}] * 3
    
    # Create GA instance
    ga_instance = pygad.GA(
        num_generations=50,
        num_parents_mating=10,
        fitness_func=fitness_volume_constrained,
        sol_per_pop=40,
        num_genes=3,
        gene_space=gene_space,
        parent_selection_type="tournament",
        crossover_type="single_point",
        mutation_type="random",
        mutation_percent_genes=30,
        random_seed=42,
        suppress_warnings=True
    )
    
    print("Running genetic algorithm...")
    ga_instance.run()
    
    # Get best solution
    solution, solution_fitness, _ = ga_instance.best_solution()
    width, length, height = solution
    
    print(f"\nBest solution found:")
    print(f"  Dimensions: {width:.3f} x {length:.3f} x {height:.3f}")
    print(f"  Fitness: {solution_fitness:.3f}")
    
    # Verify the solution
    vol, area = create_box_and_evaluate(width, length, height)
    print(f"  Volume: {vol:.3f}")
    print(f"  Surface Area: {area:.3f}")
    print(f"  Area constraint ({MAX_AREA}): {'Satisfied' if area <= MAX_AREA else 'Violated'}")
else:
    print("Skipping GA execution (PyGAD not installed)")
    print("\nTheoretical best solution for max volume with area <= 100:")
    print("  A cube with side ~4.08 gives volume ~68.0 and area ~100")

In [ ]:
if PYGAD_AVAILABLE:
    # Plot fitness over generations
    fitness_history = ga_instance.best_solutions_fitness
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=fitness_history,
        mode='lines+markers',
        name='Best Fitness',
        line=dict(color='blue', width=2),
        marker=dict(size=4)
    ))
    
    fig.update_layout(
        title="GA Optimization: Fitness Over Generations",
        xaxis_title="Generation",
        yaxis_title="Fitness (Volume)",
        width=700,
        height=400,
        plot_bgcolor='white'
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    
    fig.show()

## 2. Multi-Objective Optimization

In real-world design problems, we often need to optimize multiple conflicting objectives. This section demonstrates multi-objective optimization using NSGA-II.

In [ ]:
# 2D multi-objective optimization example
# Objective 1: Close to point A (+1.5, -1.0)
# Objective 2: Close to point B (-1.5, +1.0)
# These objectives conflict, creating a Pareto front

TARGET_A = np.array([1.5, -1.0])
TARGET_B = np.array([-1.5, 1.0])

def mo2_fitness(ga_instance, solution, solution_idx):
    """Multi-objective fitness: minimize distance to two conflicting targets."""
    x = np.array(solution, dtype=float)
    
    # Objective 1: negative squared distance to A (maximize = get closer)
    f1 = -np.sum((x - TARGET_A)**2)
    
    # Objective 2: negative squared distance to B (maximize = get closer)
    f2 = -np.sum((x - TARGET_B)**2)
    
    return [f1, f2]

# Test the fitness function
test_point = [0, 0]
f1, f2 = mo2_fitness(None, test_point, 0)
print(f"Point (0, 0):")
print(f"  Distance to A squared: {-f1:.3f}")
print(f"  Distance to B squared: {-f2:.3f}")

In [ ]:
if PYGAD_AVAILABLE:
    # Gene space: 2D coordinates in [-3, 3]
    gene_space_mo = [{"low": -3.0, "high": 3.0}] * 2
    
    # Create multi-objective GA instance
    ga_mo = pygad.GA(
        num_generations=60,
        num_parents_mating=25,
        fitness_func=mo2_fitness,
        sol_per_pop=80,
        num_genes=2,
        gene_space=gene_space_mo,
        parent_selection_type="nsga2",  # NSGA-II for multi-objective
        crossover_type="single_point",
        mutation_type="random",
        mutation_percent_genes=50,
        random_seed=42,
        suppress_warnings=True
    )
    
    print("Running multi-objective GA (NSGA-II)...")
    ga_mo.run()
    
    print(f"\nOptimization complete.")
    print(f"Population size: {len(ga_mo.population)}")
else:
    print("Skipping multi-objective GA (PyGAD not installed)")

In [ ]:
def compute_pareto_front(fitness_values):
    """Compute Pareto front indices from fitness values."""
    n = len(fitness_values)
    pareto_indices = []
    
    for i in range(n):
        is_dominated = False
        for j in range(n):
            if i != j:
                # Check if j dominates i (j is better in all objectives)
                if all(fitness_values[j][k] >= fitness_values[i][k] for k in range(len(fitness_values[i]))) and \
                   any(fitness_values[j][k] > fitness_values[i][k] for k in range(len(fitness_values[i]))):
                    is_dominated = True
                    break
        if not is_dominated:
            pareto_indices.append(i)
    
    return pareto_indices

if PYGAD_AVAILABLE:
    # Get all solutions and their fitness values
    population = ga_mo.population
    
    # Calculate fitness for all solutions
    all_fitness = []
    for sol in population:
        f = mo2_fitness(None, sol, 0)
        all_fitness.append(f)
    all_fitness = np.array(all_fitness)
    
    # Find Pareto front
    pareto_indices = compute_pareto_front(all_fitness)
    
    print(f"Pareto front size: {len(pareto_indices)}")
    print(f"\nSample Pareto-optimal solutions:")
    for i in pareto_indices[:5]:
        sol = population[i]
        f1, f2 = all_fitness[i]
        print(f"  ({sol[0]:.3f}, {sol[1]:.3f}) -> f1={f1:.3f}, f2={f2:.3f}")

In [ ]:
if PYGAD_AVAILABLE:
    # Plot Pareto front
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["Decision Space", "Objective Space"])
    
    # Decision space plot
    non_pareto = [i for i in range(len(population)) if i not in pareto_indices]
    
    # Non-Pareto points
    if non_pareto:
        fig.add_trace(go.Scatter(
            x=population[non_pareto, 0],
            y=population[non_pareto, 1],
            mode='markers',
            marker=dict(size=6, color='lightgray'),
            name='Non-Pareto'
        ), row=1, col=1)
    
    # Pareto points
    pareto_pop = population[pareto_indices]
    fig.add_trace(go.Scatter(
        x=pareto_pop[:, 0],
        y=pareto_pop[:, 1],
        mode='markers',
        marker=dict(size=10, color='darkred'),
        name='Pareto Front'
    ), row=1, col=1)
    
    # Target points
    fig.add_trace(go.Scatter(
        x=[TARGET_A[0], TARGET_B[0]],
        y=[TARGET_A[1], TARGET_B[1]],
        mode='markers+text',
        marker=dict(size=15, color=['blue', 'green'], symbol='star'),
        text=['Target A', 'Target B'],
        textposition='top center',
        name='Targets'
    ), row=1, col=1)
    
    # Objective space plot
    if non_pareto:
        fig.add_trace(go.Scatter(
            x=all_fitness[non_pareto, 0],
            y=all_fitness[non_pareto, 1],
            mode='markers',
            marker=dict(size=6, color='lightgray'),
            name='Non-Pareto',
            showlegend=False
        ), row=1, col=2)
    
    pareto_fitness = all_fitness[pareto_indices]
    # Sort Pareto front for line
    sort_idx = np.argsort(pareto_fitness[:, 0])
    fig.add_trace(go.Scatter(
        x=pareto_fitness[sort_idx, 0],
        y=pareto_fitness[sort_idx, 1],
        mode='markers+lines',
        marker=dict(size=10, color='darkred'),
        line=dict(color='darkred', width=2),
        name='Pareto Front',
        showlegend=False
    ), row=1, col=2)
    
    fig.update_xaxes(title_text="X", row=1, col=1)
    fig.update_yaxes(title_text="Y", row=1, col=1)
    fig.update_xaxes(title_text="Objective 1 (toward A)", row=1, col=2)
    fig.update_yaxes(title_text="Objective 2 (toward B)", row=1, col=2)
    
    fig.update_layout(
        title="2-Objective Pareto Front",
        width=1000,
        height=450,
        showlegend=True
    )
    
    fig.show()
else:
    # Show a conceptual plot
    fig = go.Figure()
    
    # Generate synthetic Pareto front for visualization
    t = np.linspace(0, 1, 30)
    pareto_x = TARGET_A[0] * t + TARGET_B[0] * (1 - t)
    pareto_y = TARGET_A[1] * t + TARGET_B[1] * (1 - t)
    
    fig.add_trace(go.Scatter(
        x=pareto_x, y=pareto_y,
        mode='markers+lines',
        marker=dict(size=8, color='darkred'),
        name='Conceptual Pareto Front'
    ))
    
    fig.add_trace(go.Scatter(
        x=[TARGET_A[0], TARGET_B[0]],
        y=[TARGET_A[1], TARGET_B[1]],
        mode='markers+text',
        marker=dict(size=15, color=['blue', 'green'], symbol='star'),
        text=['Target A', 'Target B'],
        textposition='top center',
        name='Targets'
    ))
    
    fig.update_layout(
        title="Conceptual Pareto Front (PyGAD not installed)",
        xaxis_title="X",
        yaxis_title="Y",
        width=600,
        height=500
    )
    fig.show()

## 3. Topology-Based Optimization: Room Layout

Let's apply GA to optimize a simple room layout problem using topologic_fast geometry.

In [ ]:
# Problem: Optimize placement of 3 rectangular rooms in a bounding box
# Objectives:
# 1. Maximize total room area
# 2. Minimize overlap between rooms
# 3. Keep rooms within bounds

BOUND_WIDTH = 20
BOUND_HEIGHT = 15

def calculate_overlap(rect1, rect2):
    """Calculate overlap area between two rectangles [x, y, w, h]."""
    x1, y1, w1, h1 = rect1
    x2, y2, w2, h2 = rect2
    
    # Calculate overlap
    ox = max(0, min(x1 + w1, x2 + w2) - max(x1, x2))
    oy = max(0, min(y1 + h1, y2 + h2) - max(y1, y2))
    
    return ox * oy

def room_layout_fitness(ga_instance, solution, solution_idx):
    """Fitness for room layout optimization."""
    # Solution: [x1, y1, w1, h1, x2, y2, w2, h2, x3, y3, w3, h3]
    rooms = [
        solution[0:4],
        solution[4:8],
        solution[8:12]
    ]
    
    total_area = 0
    total_overlap = 0
    out_of_bounds_penalty = 0
    
    for i, (x, y, w, h) in enumerate(rooms):
        # Ensure positive dimensions
        w = max(1, w)
        h = max(1, h)
        
        # Calculate area
        total_area += w * h
        
        # Check bounds
        if x < 0:
            out_of_bounds_penalty += abs(x) * 10
        if y < 0:
            out_of_bounds_penalty += abs(y) * 10
        if x + w > BOUND_WIDTH:
            out_of_bounds_penalty += (x + w - BOUND_WIDTH) * 10
        if y + h > BOUND_HEIGHT:
            out_of_bounds_penalty += (y + h - BOUND_HEIGHT) * 10
    
    # Calculate overlaps
    for i in range(len(rooms)):
        for j in range(i + 1, len(rooms)):
            total_overlap += calculate_overlap(rooms[i], rooms[j])
    
    # Fitness: maximize area, minimize overlap and out-of-bounds
    fitness = total_area - total_overlap * 5 - out_of_bounds_penalty
    
    return fitness

# Test
test_layout = [0, 0, 5, 5, 6, 0, 5, 5, 0, 6, 10, 4]
print(f"Test layout fitness: {room_layout_fitness(None, test_layout, 0)}")

In [ ]:
if PYGAD_AVAILABLE:
    # Gene space for 3 rooms: [x, y, w, h] each
    gene_space_layout = [
        # Room 1
        {"low": 0.0, "high": BOUND_WIDTH - 2},   # x1
        {"low": 0.0, "high": BOUND_HEIGHT - 2},  # y1
        {"low": 2.0, "high": 8.0},                # w1
        {"low": 2.0, "high": 8.0},                # h1
        # Room 2
        {"low": 0.0, "high": BOUND_WIDTH - 2},
        {"low": 0.0, "high": BOUND_HEIGHT - 2},
        {"low": 2.0, "high": 8.0},
        {"low": 2.0, "high": 8.0},
        # Room 3
        {"low": 0.0, "high": BOUND_WIDTH - 2},
        {"low": 0.0, "high": BOUND_HEIGHT - 2},
        {"low": 2.0, "high": 8.0},
        {"low": 2.0, "high": 8.0},
    ]
    
    ga_layout = pygad.GA(
        num_generations=100,
        num_parents_mating=20,
        fitness_func=room_layout_fitness,
        sol_per_pop=60,
        num_genes=12,
        gene_space=gene_space_layout,
        parent_selection_type="tournament",
        crossover_type="two_points",
        mutation_type="random",
        mutation_percent_genes=25,
        random_seed=42,
        suppress_warnings=True
    )
    
    print("Optimizing room layout...")
    ga_layout.run()
    
    solution, fitness, _ = ga_layout.best_solution()
    print(f"\nBest layout found (fitness: {fitness:.2f}):")
    
    rooms_result = [
        solution[0:4],
        solution[4:8],
        solution[8:12]
    ]
    for i, (x, y, w, h) in enumerate(rooms_result):
        print(f"  Room {i+1}: position=({x:.2f}, {y:.2f}), size=({w:.2f} x {h:.2f}), area={w*h:.2f}")

In [ ]:
def visualize_room_layout(rooms, bounds, title="Room Layout"):
    """Visualize room layout using topologic_fast and Plotly."""
    fig = go.Figure()
    
    colors = ['rgba(255, 99, 71, 0.6)', 'rgba(60, 179, 113, 0.6)', 'rgba(100, 149, 237, 0.6)']
    
    # Create and plot rooms using topologic_fast
    for i, (x, y, w, h) in enumerate(rooms):
        w = max(1, w)
        h = max(1, h)
        
        # Create a face for the room
        room_face = tf.Face.Rectangle(x=x + w/2, y=y + h/2, z=0, width=w, length=h)
        
        # Get vertices
        vertices = room_face.Vertices()
        xs = [v.X() for v in vertices] + [vertices[0].X()]
        ys = [v.Y() for v in vertices] + [vertices[0].Y()]
        
        fig.add_trace(go.Scatter(
            x=xs, y=ys,
            fill='toself',
            fillcolor=colors[i],
            line=dict(color='black', width=2),
            name=f'Room {i+1} ({w:.1f}x{h:.1f})',
            text=f'Room {i+1}<br>Area: {w*h:.1f}',
            hoverinfo='text'
        ))
    
    # Plot bounds
    fig.add_shape(
        type="rect",
        x0=0, y0=0, x1=bounds[0], y1=bounds[1],
        line=dict(color="black", width=3, dash="dash")
    )
    
    fig.update_layout(
        title=title,
        xaxis_title="X",
        yaxis_title="Y",
        width=700,
        height=550,
        xaxis=dict(range=[-2, bounds[0] + 2]),
        yaxis=dict(range=[-2, bounds[1] + 2], scaleanchor='x'),
        plot_bgcolor='white'
    )
    
    return fig

if PYGAD_AVAILABLE:
    fig = visualize_room_layout(rooms_result, (BOUND_WIDTH, BOUND_HEIGHT),
                                title=f"Optimized Room Layout (fitness: {fitness:.2f})")
    fig.show()
else:
    # Show example layout
    example_rooms = [(0, 0, 6, 6), (7, 0, 6, 6), (0, 7, 12, 5)]
    fig = visualize_room_layout(example_rooms, (BOUND_WIDTH, BOUND_HEIGHT),
                                title="Example Room Layout (PyGAD not installed)")
    fig.show()

## 4. Higher-Dimensional Multi-Objective Optimization

For problems with 4+ objectives, we can visualize the Pareto front using parallel coordinates.

In [ ]:
# 4-objective optimization example
# Decision vector: 4D point in [-2, 2]
# Objectives:
#   f1: Close to (+1, +1, +1, +1)
#   f2: Close to (-1, -1, -1, -1)
#   f3: Close to origin (0, 0, 0, 0)
#   f4: Far from origin (maximize magnitude)

def mo4_fitness(ga_instance, solution, solution_idx):
    """4-objective fitness function with conflicting objectives."""
    x = np.array(solution, dtype=float)
    
    # f1: peak at +1 vector
    f1 = -np.sum((x - 1.0)**2)
    
    # f2: peak at -1 vector
    f2 = -np.sum((x + 1.0)**2)
    
    # f3: peak at 0 vector
    f3 = -np.sum(x**2)
    
    # f4: prefers large magnitude (conflicts with f3)
    f4 = np.sum(x**2)
    
    return [f1, f2, f3, f4]

print("4-objective test:")
print(f"  At origin: {mo4_fitness(None, [0, 0, 0, 0], 0)}")
print(f"  At (+1, +1, +1, +1): {mo4_fitness(None, [1, 1, 1, 1], 0)}")
print(f"  At (-1, -1, -1, -1): {mo4_fitness(None, [-1, -1, -1, -1], 0)}")

In [ ]:
if PYGAD_AVAILABLE:
    gene_space_mo4 = [{"low": -2.0, "high": 2.0}] * 4
    
    ga_mo4 = pygad.GA(
        num_generations=200,
        num_parents_mating=40,
        fitness_func=mo4_fitness,
        sol_per_pop=120,
        num_genes=4,
        gene_space=gene_space_mo4,
        parent_selection_type="nsga2",
        mutation_percent_genes=25,
        random_seed=42,
        suppress_warnings=True
    )
    
    print("Running 4-objective optimization...")
    ga_mo4.run()
    
    # Get results
    pop4 = ga_mo4.population
    fitness4 = np.array([mo4_fitness(None, sol, 0) for sol in pop4])
    pareto_idx4 = compute_pareto_front(fitness4)
    
    print(f"\nPareto front size: {len(pareto_idx4)}")

In [ ]:
def plot_parallel_coordinates(fitness_values, pareto_indices, objective_names):
    """Plot multi-objective results using parallel coordinates."""
    n_objectives = fitness_values.shape[1]
    
    # Create color array (Pareto = red, non-Pareto = gray)
    colors = ['red' if i in pareto_indices else 'lightgray' for i in range(len(fitness_values))]
    color_values = [1 if i in pareto_indices else 0 for i in range(len(fitness_values))]
    
    # Sort so Pareto points are drawn on top
    sort_idx = sorted(range(len(fitness_values)), key=lambda i: color_values[i])
    
    # Create parallel coordinates dimensions
    dimensions = []
    for i, name in enumerate(objective_names):
        dimensions.append(dict(
            range=[fitness_values[:, i].min(), fitness_values[:, i].max()],
            label=name,
            values=fitness_values[sort_idx, i]
        ))
    
    fig = go.Figure(data=go.Parcoords(
        line=dict(
            color=[color_values[i] for i in sort_idx],
            colorscale=[[0, 'lightgray'], [1, 'red']],
            showscale=False
        ),
        dimensions=dimensions
    ))
    
    fig.update_layout(
        title="4-Objective Pareto Front (Parallel Coordinates)",
        width=1000,
        height=500
    )
    
    return fig

if PYGAD_AVAILABLE:
    objective_names = ["Toward +1", "Toward -1", "Toward 0", "Large Magnitude"]
    fig = plot_parallel_coordinates(fitness4, pareto_idx4, objective_names)
    fig.show()
else:
    print("Parallel coordinates plot requires PyGAD results.")
    print("\nConceptually, parallel coordinates show each objective as a vertical axis.")
    print("Each solution is a polyline connecting its values across all objectives.")
    print("Pareto-optimal solutions (red) represent non-dominated trade-offs.")

## 5. Summary

In this notebook, we demonstrated:

1. **Single-objective optimization**: Maximizing box volume with surface area constraints using topologic_fast geometry

2. **Multi-objective optimization**: Using NSGA-II to find Pareto-optimal solutions for conflicting objectives

3. **Layout optimization**: Optimizing room placement using GA with topologic_fast for visualization

4. **High-dimensional visualization**: Using parallel coordinates for 4+ objective problems

### Key Points

| Feature | topologicpy | topologic_fast |
|---------|-------------|----------------|
| GA Class | `GA` wrapper for PyGAD | Use PyGAD directly |
| Pareto Analysis | Built-in methods | Manual or scipy |
| Geometry Creation | Same API | Same API |
| Performance | Python | Rust (faster) |

topologic_fast provides high-performance geometry operations that integrate seamlessly with Python optimization libraries like PyGAD, scipy.optimize, or DEAP.

### Applications

- Building energy optimization
- Space planning and layout
- Structural design optimization
- Form-finding in architecture

In [ ]:
# Clean up
tf.clear_store()
print("Topology store cleared.")